In [ ]:
!pip install transformers datasets accelerate
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from google.colab import userdata
import os

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN loaded from Colab secrets and set as environment variable.")
# Now, when load_dataset is called, it will automatically use this token for authentication.

HF_TOKEN loaded from Colab secrets and set as environment variable.


In [ ]:
# ============================================================
# Phase 1: CinePile Text-Only Evaluation — Final Fix
# answer_key 是答案全文，需与 choices 匹配转为 A~E
# ============================================================

import gc
import torch
import pandas as pd
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
    BitsAndBytesConfig
)
from tqdm import tqdm



In [ ]:

# ============================================================
# 1. Configure Class
# ============================================================
@dataclass
class ExperimentConfig:
    dataset_name: str           = 'tomg-group-umd/cinepile'
    dataset_split: str          = 'test'
    max_samples: Optional[int]  = 200      # 调试时设 200

    max_scene_length: int       = 1500
    max_input_tokens: int       = 2048
    deberta_max_tokens: int     = 512

    max_new_tokens: int         = 5
    do_sample: bool             = False

    load_in_4bit: bool          = True
    bnb_4bit_quant_type: str    = "nf4"
    bnb_4bit_compute_dtype: torch.dtype = torch.bfloat16

    models: list = field(default_factory=lambda: [
        ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
        ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
        ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
    ])

    output_csv: str = 'phase1_results.csv'

    category_map: dict = field(default_factory=lambda: {
        'TEMP': 'Temporal',
        'CRD':  'Character and',        # map 'Character and\nRelationship Dynamics'
        'NPA':  'Narrative and',        # map 'Narrative and\nPlot Analysis'
        'STA':  'Setting and',          # map 'Setting and\nTechnical Analysis'
        'TH':   'Theme Exploration',    # map 'Theme Exploration'
    })




In [ ]:

# ============================================================
# 2. Prepare dataset
# ============================================================
class CinePileDataset:
    """
    CinePile 的 answer_key 是答案全文字符串。
    通过与 choices 列表匹配，转换为字母索引 'A'~'E'。
    """
    LETTERS = ['A', 'B', 'C', 'D', 'E']

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.data   = self._load()

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        """
        将答案全文映射到对应字母。
        先做精确匹配，再做 strip 后的模糊匹配，
        均失败时返回 'A' 并打印警告。
        """
        # 精确匹配
        for i, choice in enumerate(choices):
            if answer_text == choice:
                return cls.LETTERS[i]

        # strip 后匹配（防止空格差异）
        answer_stripped = answer_text.strip()
        for i, choice in enumerate(choices):
            if answer_stripped == choice.strip():
                return cls.LETTERS[i]

        # 包含匹配（防止部分截断）
        for i, choice in enumerate(choices):
            if answer_stripped in choice or choice.strip() in answer_stripped:
                return cls.LETTERS[i]

        print(f"  [Warning] Cannot match answer_key to choices.")
        print(f"    answer_key : {repr(answer_text)}")
        print(f"    choices    : {choices}")
        return 'A'  # fallback

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == 'true'

    def _load(self):
        raw = load_dataset(
            self.config.dataset_name,
            split=self.config.dataset_split
        )
        samples = []
        unmatched = 0

        for ex in raw:
            choices    = ex['choices']
            answer_key = ex['answer_key']

            # 核心修复：answer_key 全文 → 字母
            letter = self._answer_text_to_letter(answer_key, choices)
            if letter == 'A' and answer_key != choices[0]:
                unmatched += 1

            samples.append({
                'movie_scene':       ex['movie_scene'],
                'question':          ex['question'],
                'choices':           choices,
                'answer_key':        letter,           # 统一为 'A'~'E'
                'answer_text':       answer_key,       # 保留原始全文（调试用）
                'question_category': ex['question_category'],
                'hard_split':        self._normalize_hard_split(ex['hard_split']),
            })

        if self.config.max_samples:
            samples = samples[:self.config.max_samples]

        print(f"[Dataset] Loaded {len(samples)} samples")
        print(f"          answer_key sample : "
              f"{repr(samples[0]['answer_text'])} → {samples[0]['answer_key']}")
        print(f"          hard_split sample : {samples[0]['hard_split']}")
        if unmatched > 0:
            print(f"  [Warning] {unmatched} samples fell back to 'A' "
                  f"(answer not matched in choices)")
        return samples

    def run_sanity_check(self, n: int = 5):
        """快速验证前 n 个样本的 answer_key 转换是否正确"""
        print("\n=== Sanity Check ===")
        for i, s in enumerate(self.data[:n]):
            idx = ord(s['answer_key']) - 65
            matched_text = s['choices'][idx]
            match_ok = matched_text.strip() == s['answer_text'].strip()
            print(f"  [{i}] {s['answer_key']} → \"{matched_text}\"")
            print(f"       original: \"{s['answer_text']}\"  match={match_ok}")
        print()



In [ ]:

# ============================================================
# 3. Base Evaluator
# ============================================================
class BaseEvaluator(ABC):
    def __init__(self, model_id: str, config: ExperimentConfig):
        self.model_id  = model_id
        self.config    = config
        self.model     = None
        self.tokenizer = None

    @abstractmethod
    def load_model(self): pass

    @abstractmethod
    def predict(self, sample: dict) -> str: pass

    def evaluate(self, data: list) -> pd.DataFrame:
        records = []
        for sample in tqdm(data, desc=f"  {self.model_id}"):
            pred = self.predict(sample)
            records.append({
                'pred':              pred,
                'label':             sample['answer_key'],
                'question_category': sample['question_category'],
                'hard_split':        sample['hard_split'],
            })
        return pd.DataFrame(records)

    def free_memory(self):
        if self.model is not None:
            del self.model
            self.model = None
        if self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [Memory freed] {self.model_id}")


In [ ]:


# ============================================================
# 4. LLM Evaluator
# ============================================================
class LLMEvaluator(BaseEvaluator):
    def load_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = self.config.load_in_4bit,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = self.config.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype    = self.config.bnb_4bit_compute_dtype,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        self.model.eval()
        print(f"  [Loaded] {self.model_id} (4-bit)")

    def _build_prompt(self, sample: dict) -> str:
        scene   = sample['movie_scene'][:self.config.max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer with only the letter (A, B, C, D, or E):"
        )

    def predict(self, sample: dict) -> str:
        prompt = self._build_prompt(sample)
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_input_tokens,
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=self.config.do_sample,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for char in decoded:
            if char in 'ABCDE':
                return char
        return 'A'  # fallback


In [ ]:


# ============================================================
# 5. DeBERTa-v3 Evaluator
# ============================================================
class DeBERTaEvaluator(BaseEvaluator):
    """
    用 AutoModel + [CLS] embedding norm 对每个 choice 打分，
    选分数最高的 choice 作为预测答案。
    """
    def load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model     = AutoModel.from_pretrained(self.model_id).to(
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.model.eval()
        print(f"  [Loaded] {self.model_id}")

    def _score(self, text_a: str, text_b: str) -> float:
        inputs = self.tokenizer(
            text_a, text_b,
            return_tensors='pt',
            truncation=True,
            max_length=self.config.deberta_max_tokens,
            padding=True,
        ).to(self.model.device)
        with torch.no_grad():
            cls_emb = self.model(**inputs).last_hidden_state[:, 0, :]
        return cls_emb.norm(dim=-1).item()

    def predict(self, sample: dict) -> str:
        scene_q = (
            f"{sample['question']} "
            f"{sample['movie_scene'][:self.config.deberta_max_tokens // 2]}"
        )
        scores = [self._score(scene_q, c) for c in sample['choices']]
        return chr(65 + scores.index(max(scores))) # 'A'~'E'



In [ ]:

# ============================================================
# 6. Metrics calculation
# ============================================================
class MetricsCalculator:
    def __init__(self, config: ExperimentConfig):
        self.config = config

    def compute(self, df: pd.DataFrame) -> dict:
        metrics = {}
        correct = df['pred'] == df['label']

        metrics['Overall']   = correct.mean()
        metrics['Overall_n'] = len(df)

        for abbr, cat_name in self.config.category_map.items():
            mask = df['question_category'].str.contains(
                cat_name, case=False, na=False
            )
            n = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f'{abbr}_n'] = n

        hard_mask = df['hard_split'] == True
        n_hard    = int(hard_mask.sum())
        metrics['Hard']   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics['Hard_n'] = n_hard

        return metrics



In [ ]:

# ============================================================
# 7. Phase1 Runner
# ============================================================
class Phase1Runner:
    EVALUATOR_REGISTRY = {
        'llm':     LLMEvaluator,
        'deberta': DeBERTaEvaluator,
    }

    def __init__(self, config: ExperimentConfig):
        self.config       = config
        self.dataset      = CinePileDataset(config)
        self.metrics_calc = MetricsCalculator(config)
        self.all_metrics  = {}

        # 运行 sanity check，确认 answer_key 转换正确
        self.dataset.run_sanity_check(n=3)

    def run(self):
        print("="*60)
        print("PHASE 1: CinePile Text-Only Evaluation")
        print("="*60)

        for display_name, model_id, eval_type in self.config.models:
            print(f"\n>>> [{eval_type.upper()}] {display_name}")

            evaluator_cls = self.EVALUATOR_REGISTRY[eval_type]
            evaluator     = evaluator_cls(model_id, self.config)
            evaluator.load_model()

            results_df = evaluator.evaluate(self.dataset.data)

            metrics = self.metrics_calc.compute(results_df)
            self.all_metrics[display_name] = metrics

            hard_str = (f"{metrics['Hard']:.2%}"
                        if metrics.get('Hard') is not None else "N/A")
            print(f"  Overall: {metrics['Overall']:.2%}  "
                  f"| Hard: {hard_str}  "
                  f"| n={metrics['Overall_n']}")

            # Free memory after each model run
            evaluator.free_memory()

        self._display_and_save()

    def _display_and_save(self):
        display_cols = ['Overall', 'TEMP', 'CRD', 'NPA', 'STA', 'TH', 'Hard']
        rows = []
        for model_name, metrics in self.all_metrics.items():
            row = {'Model': model_name}
            for col in display_cols:
                val = metrics.get(col)
                n   = metrics.get(f'{col}_n', metrics.get('Overall_n', ''))
                row[col] = f"{val:.1%}(n={n})" if val is not None else 'N/A'
            rows.append(row)

        df = pd.DataFrame(rows)
        print("\n" + "="*60)
        print("FINAL RESULTS — CinePile Text-Only")
        print("="*60)
        print(df.to_string(index=False))
        df.to_csv(self.config.output_csv, index=False)
        print(f"\nSaved → {self.config.output_csv}")


if __name__ == '__main__':

    config = ExperimentConfig(
        max_samples     = None,     # using None for all test datasets
        max_scene_length= 1500,
        load_in_4bit    = True,
        output_csv      = 'phase1_results.csv',
        models=[
            # ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
            # ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
            ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
        ]
    )

    runner = Phase1Runner(config)
    runner.run()

README.md: 0.00B [00:00, ?B/s]

v2/train-00000-of-00003.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

v2/train-00001-of-00003.parquet:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

v2/train-00002-of-00003.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

v2/test-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 诊断代码：直接粘贴到 Colab 运行
# ============================================================
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# Step 1: 查看原始字段
sample = dataset[0]
print("=== 所有字段名 ===")
print(list(sample.keys()))

print("\n=== answer_key 值和类型 ===")
print(repr(sample['answer_key']))
print(type(sample['answer_key']))

print("\n=== choices 值和类型 ===")
print(sample['choices'])
print(type(sample['choices'][0]))

print("\n=== question_category 示例 ===")
print(repr(sample['question_category']))

print("\n=== hard_split 值和类型 ===")
print(repr(sample['hard_split']))
print(type(sample['hard_split']))

# Step 2: 查看前5个样本的 answer_key
print("\n=== 前5个样本的 answer_key ===")
for i in range(5):
    print(f"  [{i}] answer_key = {repr(dataset[i]['answer_key'])}")

# Step 3: 查看所有 answer_key 的唯一值
print("\n=== answer_key 的所有唯一值（前100个样本）===")
unique_keys = set(dataset[i]['answer_key'] for i in range(100))
print(unique_keys)

# Step 4: 查看 question_category 的唯一值
print("\n=== question_category 的唯一值（前100个样本）===")
unique_cats = set(dataset[i]['question_category'] for i in range(100))
for c in sorted(unique_cats):
    print(f"  {repr(c)}")

# Step 5: 手动测试 DeBERTa predict 一次
print("\n=== DeBERTa 手动预测测试 ===")
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base',
    num_labels=1
)
model.eval()

test_sample = dataset[0]
scores = []
scene_q = f"{test_sample['question']} {test_sample['movie_scene'][:512]}"

for i, choice in enumerate(test_sample['choices']):
    inputs = tokenizer(
        scene_q, choice,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        logit = model(**inputs).logits.squeeze().item()
    scores.append(logit)
    print(f"  Choice {chr(65+i)}: logit = {logit:.4f}")

pred = chr(65 + scores.index(max(scores)))
label = test_sample['answer_key']
print(f"\n  Predicted: {pred}")
print(f"  Label:     {repr(label)}")
print(f"  Match:     {pred == label}")


In [ ]:
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# 查看前20个 answer_key 的原始值
print("=== 前20个 answer_key ===")
for i in range(20):
    ak = dataset[i]['answer_key']
    print(f"  [{i}] type={type(ak).__name__}, value={repr(ak)}")

# 查看所有唯一值
print("\n=== 所有唯一 answer_key 值（完整测试集）===")
from collections import Counter
counter = Counter(str(dataset[i]['answer_key']) for i in range(len(dataset)))
for val, cnt in sorted(counter.items()):
    print(f"  {repr(val):10s} → {cnt} 条")


=== 前20个 answer_key ===
  [0] type=str, value='From anxiety to excitement'
  [1] type=str, value='Suggests next steps'
  [2] type=str, value='The rough foliage'
  [3] type=str, value='She closes her eyes and focuses on staying calm without moving at all.'
  [4] type=str, value='Rough and dense'
  [5] type=str, value='They hear an aircraft approaching.'
  [6] type=str, value='To reach a specific location'
  [7] type=str, value='They become more confident'
  [8] type=str, value='The sound of a helicopter approaching'
  [9] type=str, value='Reed'
  [10] type=str, value='The sound of a helicopter'
  [11] type=str, value='In a rocky plateau'
  [12] type=str, value='It is a valley with spots of light'
  [13] type=str, value='They use a thermal suit'
  [14] type=str, value='They are trying to remain undetected by the helicopter using thermal suits.'
  [15] type=str, value='It illuminates the surrounding foliage.'
  [16] type=str, value='Directly underneath'
  [17] type=str, value='Colleagues'

In [ ]:
# 运行这段诊断，查看实际的 question_category 值分布
from datasets import load_dataset
from collections import Counter

# dataset = load_dataset('tomg-group-umd/cinepile', split='test')

counter = Counter(dataset[i]['question_category'] for i in range(200))
print("=== question_category 实际值分布 ===")
for val, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    print(f"  {cnt:4d}  {repr(val)}")


=== question_category 实际值分布 ===
    88  'Character and\nRelationship Dynamics'
    63  'Setting and\nTechnical Analysis'
    24  'Temporal'
    13  'Narrative and\nPlot Analysis'
    12  'Theme Exploration'


In [ ]:
# 快速验证（无需重新加载数据集）
test_categories = [
    'Character and\nRelationship Dynamics',
    'Setting and\nTechnical Analysis',
    'Temporal',
    'Narrative and\nPlot Analysis',
    'Theme Exploration',
]

category_map = {
    'TEMP': 'Temporal',
    'CRD':  'Character and',
    'NPA':  'Narrative and',
    'STA':  'Setting and',
    'TH':   'Theme Exploration',
}

import pandas as pd
df_test = pd.DataFrame({'question_category': test_categories})

print("=== 映射验证 ===")
for abbr, keyword in category_map.items():
    mask = df_test['question_category'].str.contains(keyword, case=False, na=False)
    matched = df_test.loc[mask, 'question_category'].tolist()
    print(f"  {abbr} ({keyword!r:20s}) → {matched}")


=== 映射验证 ===
  TEMP ('Temporal'          ) → ['Temporal']
  CRD ('Character and'     ) → ['Character and\nRelationship Dynamics']
  NPA ('Narrative and'     ) → ['Narrative and\nPlot Analysis']
  STA ('Setting and'       ) → ['Setting and\nTechnical Analysis']
  TH ('Theme Exploration' ) → ['Theme Exploration']
